# 05 — Final Roman Urdu model: transformer + TF-IDF (saved for the app)

1. Fine-tune **XLM-R large** (default) — the one remaining option likely to add accuracy.
2. Train the **TF-IDF** model (character + word n-grams, same as notebook 04).
3. Combine them (weight chosen on validation) and compare with notebook 04, per source.
4. **Save both** to `models/roman_urdu_final/` with a config file, ready for the backend and a GitHub release.

If XLM-R large is not better than base, set `MODEL_NAME = "xlm-roberta-base"` in section 0 and run again: you still get a saved final model.

Same fixed split as `01_make_splits.ipynb`. No article text is written to the output files.
**Runtime:** XLM-R large ≈ 45–60 min on the RTX A4000; base ≈ 11 min.

## 0. Settings

In [1]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
from pathlib import Path

DATA_PATH      = Path(r"E:\KhabarNazar_text\data\khabarnazar_roman_urdu_dataset.csv")
SPLIT_IDS_PATH = Path(r"E:\KhabarNazar_text\data\split_ids.csv")
RESULTS_04     = Path(r"E:\KhabarNazar_text\outputs\04_seeds_muril_ensemble\results_04.csv")
OUT            = Path(r"E:\KhabarNazar_text\outputs\05_final_model")
FINAL_DIR      = Path(r"E:\KhabarNazar_text\models\roman_urdu_final")

MODEL_NAME = "xlm-roberta-large"     # or "xlm-roberta-base"
SEED       = 21
LARGE      = "large" in MODEL_NAME

# large: smaller learning rate (more stable), batch 4 x 4 = effective 16, gradient checkpointing to fit in 16 GB
MAX_LEN    = 512
BATCH_SIZE = 4 if LARGE else 16
GRAD_ACCUM = 4 if LARGE else 1
LR         = 8e-6 if LARGE else 2e-5
EPOCHS, WARMUP_RATIO, WEIGHT_DECAY, PATIENCE = 5, 0.1, 0.01, 2
TFIDF_C    = 2      # best C found in notebook 04

LABEL2ID = {"Real": 0, "Fake": 1}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
TAG = MODEL_NAME.replace("xlm-roberta-", "xlmr_")
OUT.mkdir(parents=True, exist_ok=True); FINAL_DIR.mkdir(parents=True, exist_ok=True)
print("Model:", MODEL_NAME, "| data:", DATA_PATH.exists(), "| split:", SPLIT_IDS_PATH.exists())

Model: xlm-roberta-large | data: True | split: True


## 1. GPU check

In [2]:
import torch, transformers, sklearn, platform
print("PyTorch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
print("transformers", transformers.__version__, "| sklearn", sklearn.__version__)
NUM_WORKERS = 0 if platform.system() == "Windows" else 2

PyTorch 2.11.0+cu128 | CUDA: True | NVIDIA RTX A4000
transformers 5.20.2 | sklearn 1.9.1


## 2. Data and the fixed split

In [3]:
import numpy as np, pandas as pd

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")[["id", "roman_text", "label", "source_dataset"]]
df = df.rename(columns={"roman_text": "text", "source_dataset": "source"})
df["text"] = df["text"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
df["label"] = df["label"].astype(str).str.strip().str.capitalize().map(LABEL2ID).astype(int)
df = df.merge(pd.read_csv(SPLIT_IDS_PATH), on="id", how="left", validate="one_to_one")
assert df["split"].notna().all()
train_df = df[df.split == "train"].reset_index(drop=True)
val_df   = df[df.split == "val"].reset_index(drop=True)
test_df  = df[df.split == "test"].reset_index(drop=True)
SOURCES = sorted(df.source.unique())
print("train / val / test:", len(train_df), len(val_df), len(test_df), "(expected 6615 / 826 / 826)")

train / val / test: 6615 826 826 (expected 6615 / 826 / 826)


## 3. Evaluation helpers

In [4]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

def scores(frame, p_fake, thr=0.5):
    pred = (np.asarray(p_fake) >= thr).astype(int); y = frame["label"].to_numpy()
    out = {"accuracy": accuracy_score(y, pred), "ALL": f1_score(y, pred, average="macro")}
    for s in SOURCES:
        m = (frame["source"] == s).to_numpy()
        out[s] = f1_score(y[m], pred[m], average="macro")
    return out

def bootstrap_diff(p_a, p_b, frame, n=2000, seed=0):
    rng = np.random.default_rng(seed); y = frame.label.to_numpy()
    a, b = (p_a >= .5).astype(int), (p_b >= .5).astype(int); d = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        d.append(f1_score(y[i], a[i], average="macro") - f1_score(y[i], b[i], average="macro"))
    return np.mean(d), np.percentile(d, [2.5, 97.5])

## 4. TF-IDF model (fast, ~1 min)

In [5]:
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline, make_union
from sklearn.linear_model import LogisticRegression

tfidf = make_pipeline(
    make_union(
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True, lowercase=True),
        TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True, lowercase=True)),
    LogisticRegression(C=TFIDF_C, max_iter=3000, class_weight="balanced"))
tfidf.fit(train_df.text, train_df.label)
tf_val, tf_test = tfidf.predict_proba(val_df.text)[:, 1], tfidf.predict_proba(test_df.text)[:, 1]
print("TF-IDF test macro-F1:", round(scores(test_df, tf_test)["ALL"], 4))

TF-IDF test macro-F1: 0.8158


## 5. Fine-tune the transformer

If predictions for this model already exist (e.g. after a power cut), training is skipped and the saved model is reused.
**Watch the first epochs:** if validation accuracy stays at ~0.52 for 2 epochs, the run has collapsed (predicts one class) — stop it, change `SEED` in section 0 and run all again.

In [6]:
import gc, math, shutil, time
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding,
                          TrainingArguments, Trainer, EarlyStoppingCallback, set_seed)

TR_DIR = FINAL_DIR / "transformer"
probs_file = OUT / f"probs_{TAG}_s{SEED}.csv"

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
enc = lambda d: Dataset.from_pandas(d[["text", "label"]]).map(
    lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN), batched=True, remove_columns=["text"])

def hf_metrics(eval_pred):
    logits, labels = eval_pred; preds = logits.argmax(-1)
    return {"accuracy": accuracy_score(labels, preds), "f1_macro": f1_score(labels, preds, average="macro")}

if probs_file.exists() and (TR_DIR / "config.json").exists():
    p = pd.read_csv(probs_file)
    tr_val = p[p.split == "val"].p_fake.to_numpy(); tr_test = p[p.split == "test"].p_fake.to_numpy()
    print("Already trained — loaded saved predictions from", probs_file.name)
else:
    set_seed(SEED)
    tr, va, te = enc(train_df), enc(val_df), enc(test_df)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2,
                                                               id2label=ID2LABEL, label2id=LABEL2ID)
    steps = math.ceil(len(tr) / (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS
    ckpt = OUT / f"checkpoints_{TAG}"
    args = TrainingArguments(
        output_dir=str(ckpt), num_train_epochs=EPOCHS, learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=16,
        gradient_accumulation_steps=GRAD_ACCUM, gradient_checkpointing=LARGE,
        warmup_steps=int(WARMUP_RATIO * steps), weight_decay=WEIGHT_DECAY, lr_scheduler_type="linear",
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1, save_only_model=True,
        load_best_model_at_end=True, metric_for_best_model="f1_macro", greater_is_better=True,
        logging_steps=50, bf16=torch.cuda.is_bf16_supported(), dataloader_num_workers=NUM_WORKERS,
        report_to="none", seed=SEED)
    trainer = Trainer(model=model, args=args, train_dataset=tr, eval_dataset=va, processing_class=tok,
                      data_collator=DataCollatorWithPadding(tok), compute_metrics=hf_metrics,
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
    t0 = time.time(); trainer.train()
    prob = lambda ds: torch.softmax(torch.tensor(trainer.predict(ds).predictions).float(), -1)[:, 1].numpy()
    tr_val, tr_test = prob(va), prob(te)
    print(f"Training time {(time.time()-t0)/60:.1f} min")

    if scores(val_df, tr_val)["ALL"] < 0.5:
        print("\n*** The run collapsed (predicts one class). Change SEED in section 0 and run all again. ***")
    else:
        trainer.save_model(str(TR_DIR)); tok.save_pretrained(str(TR_DIR))
        pd.concat([pd.DataFrame({"id": val_df.id, "split": "val", "p_fake": tr_val}),
                   pd.DataFrame({"id": test_df.id, "split": "test", "p_fake": tr_test})]).to_csv(probs_file, index=False)
        pd.DataFrame(trainer.state.log_history).to_csv(OUT / f"log_{TAG}_s{SEED}.csv", index=False)
        print("Saved transformer to", TR_DIR)
    del trainer, model; gc.collect(); torch.cuda.empty_cache()
    shutil.rmtree(ckpt, ignore_errors=True)

print("Transformer val macro-F1:", round(scores(val_df, tr_val)["ALL"], 4),
      "| test macro-F1:", round(scores(test_df, tr_test)["ALL"], 4))

W1022 21:41:47.182000 22260 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Map:   0%|          | 0/6615 [00:00<?, ? examples/s]

Map:   0%|          | 0/826 [00:00<?, ? examples/s]

Map:   0%|          | 0/826 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training time 38.2 min


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved transformer to E:\KhabarNazar_text\models\roman_urdu_final\transformer
Transformer val macro-F1: 0.8654 | test macro-F1: 0.8422


## 6. Combine transformer + TF-IDF (weight chosen on validation only)

In [7]:
grid = np.linspace(0, 1, 21)
W_TFIDF = float(max(grid, key=lambda w: scores(val_df, (1 - w) * tr_val + w * tf_val)["ALL"]))
comb_val, comb_test = (1 - W_TFIDF) * tr_val + W_TFIDF * tf_val, (1 - W_TFIDF) * tr_test + W_TFIDF * tf_test
print(f"TF-IDF weight: {W_TFIDF:.2f}  (transformer weight {1 - W_TFIDF:.2f})")

rows = [{"model": "TF-IDF char+word + LR", **scores(test_df, tf_test)},
        {"model": f"{MODEL_NAME} (seed {SEED})", **scores(test_df, tr_test)},
        {"model": f"{MODEL_NAME} + TF-IDF  <- FINAL", **scores(test_df, comb_test)}]
final_table = pd.DataFrame(rows)[["model", "accuracy", "ALL"] + SOURCES]
final_table.to_csv(OUT / f"results_05_{TAG}.csv", index=False)
print("Test macro-F1 on the fixed 826-article split:")
display(final_table.round(3))

if RESULTS_04.exists():
    r4 = pd.read_csv(RESULTS_04)
    keep = r4[r4.model.str.contains("mean of|ensemble|TF-IDF", regex=True)][["model", "accuracy", "ALL"] + SOURCES]
    print("For comparison, notebook 04:")
    display(keep.round(3))

for name, p in [(MODEL_NAME, tr_test), ("FINAL (combined)", comb_test)]:
    d, (lo, hi) = bootstrap_diff(p, tf_test, test_df)
    verdict = "better" if lo > 0 else ("worse" if hi < 0 else "not clearly different")
    print(f"{name:22s} vs TF-IDF: {d:+.3f} macro-F1 (95% CI {lo:+.3f} to {hi:+.3f}) -> {verdict}")

print("\nFINAL model, test set:")
print(classification_report(test_df.label, (comb_test >= .5).astype(int), target_names=["Real", "Fake"], digits=4))

TF-IDF weight: 0.55  (transformer weight 0.45)
Test macro-F1 on the fixed 826-article split:


,model,accuracy,ALL,ax_to_grind,bend_the_truth,fire,hook_and_bait
0,TF-IDF char+word + LR,0.816,0.816,0.801,0.658,0.620,0.883
1,xlm-roberta-large (seed 21),0.843,0.842,0.825,0.589,0.741,0.914
2,xlm-roberta-large + TF-IDF <- FINAL,0.846,0.846,0.825,0.612,0.707,0.921


For comparison, notebook 04:


,model,accuracy,ALL,ax_to_grind,bend_the_truth,fire,hook_and_bait
0,TF-IDF char+word + LR,0.816,0.816,0.801,0.658,0.620,0.883
4,xlmr mean of 3 seeds,0.827,0.826,0.806,0.602,0.695,0.897
5,xlmr ensemble (3 seeds),0.837,0.836,0.818,0.614,0.688,0.907
9,muril mean of 3 seeds,0.818,0.818,0.818,0.592,0.682,0.877
10,muril ensemble (3 seeds),0.831,0.830,0.842,0.573,0.663,0.893
11,XLM-R + MuRIL ensemble,0.837,0.836,0.838,0.606,0.701,0.895
12,Transformers + TF-IDF (w_tfidf=0.50),0.844,0.843,0.838,0.628,0.662,0.912


xlm-roberta-large      vs TF-IDF: +0.026 macro-F1 (95% CI -0.003 to +0.054) -> not clearly different
FINAL (combined)       vs TF-IDF: +0.030 macro-F1 (95% CI +0.004 to +0.057) -> better

FINAL model, test set:
              precision    recall  f1-score   support

        Real     0.8378    0.8712    0.8542       427
        Fake     0.8560    0.8195    0.8374       399

    accuracy                         0.8462       826
   macro avg     0.8469    0.8454    0.8458       826
weighted avg     0.8466    0.8462    0.8461       826



## 7. Save the final model (transformer + TF-IDF + config)

In [8]:
import json, datetime
joblib.dump(tfidf, FINAL_DIR / "tfidf.joblib")
config = {
    "name": "KhabarNazar Roman Urdu fake news model",
    "transformer": "transformer/", "transformer_base": MODEL_NAME, "tfidf": "tfidf.joblib",
    "weight_tfidf": W_TFIDF, "weight_transformer": 1 - W_TFIDF, "threshold": 0.5,
    "max_len": MAX_LEN, "label2id": LABEL2ID, "positive_class": "Fake",
    "text_cleaning": "strip, collapse whitespace (no other changes)",
    "split": "split_ids.csv from 01_make_splits.ipynb", "seed": SEED,
    "test_macro_f1": {k: round(float(v), 4) for k, v in scores(test_df, comb_test).items()},
    "versions": {"transformers": transformers.__version__, "torch": torch.__version__, "sklearn": sklearn.__version__},
    "created": datetime.date.today().isoformat(),
}
(FINAL_DIR / "final_config.json").write_text(json.dumps(config, indent=2))
for p in sorted(FINAL_DIR.rglob("*")):
    if p.is_file(): print(f"{p.stat().st_size/1e6:9.1f} MB  {p.relative_to(FINAL_DIR)}")

      0.0 MB  final_config.json
      4.4 MB  tfidf.joblib
      0.0 MB  transformer\config.json
   2239.6 MB  transformer\model.safetensors
     17.1 MB  transformer\tokenizer.json
      0.0 MB  transformer\tokenizer_config.json
      0.0 MB  transformer\training_args.bin


## 8. Test the saved model the way the backend would use it

Loads everything from `models/roman_urdu_final/` (nothing from memory) and classifies a few test articles.

In [9]:
def load_final(folder):
    folder = Path(folder); cfg = json.loads((folder / "final_config.json").read_text())
    t = AutoTokenizer.from_pretrained(str(folder / cfg["transformer"]))
    m = AutoModelForSequenceClassification.from_pretrained(str(folder / cfg["transformer"])).eval()
    dev = "cuda" if torch.cuda.is_available() else "cpu"; m.to(dev)
    tf = joblib.load(folder / cfg["tfidf"])
    def predict(texts):
        texts = [" ".join(str(x).split()) for x in texts]
        b = t(texts, truncation=True, max_length=cfg["max_len"], padding=True, return_tensors="pt").to(dev)
        with torch.no_grad():
            p_tr = torch.softmax(m(**b).logits.float(), -1)[:, 1].cpu().numpy()
        p = cfg["weight_transformer"] * p_tr + cfg["weight_tfidf"] * tf.predict_proba(texts)[:, 1]
        return [("Fake" if x >= cfg["threshold"] else "Real", round(float(x), 3)) for x in p]
    return predict

predict = load_final(FINAL_DIR)
sample = test_df.sample(6, random_state=3)
for (lab, p), (_, r) in zip(predict(sample.text), sample.iterrows()):
    print(f"pred={lab:4s} p_fake={p:.3f} | true={ID2LABEL[r.label]:4s} | {r.source}")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

pred=Real p_fake=0.092 | true=Real | hook_and_bait
pred=Fake p_fake=0.927 | true=Fake | hook_and_bait
pred=Real p_fake=0.308 | true=Real | ax_to_grind
pred=Fake p_fake=0.969 | true=Fake | ax_to_grind
pred=Real p_fake=0.311 | true=Fake | fire
pred=Real p_fake=0.163 | true=Fake | ax_to_grind


In [10]:
# 9. Release copy in half precision (fits GitHub's 2 GB file limit) + check it gives the same results
REL_DIR = Path(r"E:\KhabarNazar_text\models\roman_urdu_release")
REL_DIR.mkdir(parents=True, exist_ok=True)

full = AutoModelForSequenceClassification.from_pretrained(str(TR_DIR), dtype=torch.float32)
full.half().save_pretrained(str(REL_DIR))
AutoTokenizer.from_pretrained(str(TR_DIR)).save_pretrained(str(REL_DIR))
del full; gc.collect(); torch.cuda.empty_cache()

# Load it exactly like the backend does (no dtype argument) and score the test set again
rel_tok = AutoTokenizer.from_pretrained(str(REL_DIR))
rel = AutoModelForSequenceClassification.from_pretrained(str(REL_DIR)).eval().to("cuda")
print("Loaded dtype:", next(rel.parameters()).dtype, "| labels:", rel.config.id2label)
probs = []
with torch.no_grad():
    for i in range(0, len(test_df), 16):
        b = rel_tok(list(test_df.text[i:i+16]), truncation=True, max_length=MAX_LEN,
                    padding=True, return_tensors="pt").to("cuda")
        probs.extend(torch.softmax(rel(**b).logits.float(), -1)[:, 1].cpu().numpy())
probs = np.array(probs)
print("Original test macro-F1:", round(scores(test_df, tr_test)["ALL"], 4))
print("Release  test macro-F1:", round(scores(test_df, probs)["ALL"], 4))
print("Largest probability difference:", round(float(np.abs(probs - tr_test).max()), 4))
for p in sorted(REL_DIR.iterdir()):
    print(f"{p.stat().st_size/1e6:9.1f} MB  {p.name}")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loaded dtype: torch.float16 | labels: {0: 'Real', 1: 'Fake'}
Original test macro-F1: 0.8422
Release  test macro-F1: 0.8434
Largest probability difference: 0.1061
      0.0 MB  config.json
   1119.8 MB  model.safetensors
     17.1 MB  tokenizer.json
      0.0 MB  tokenizer_config.json
